<!-- # Environment Setup: Clone and Install Editable Ultralytics -->


In [ ]:
# Clone the Ultralytics repository into the Kaggle working directory.

!git clone https://github.com/ultralytics/ultralytics.git

In [ ]:
# Switch the notebook working directory to the cloned Ultralytics repository.

%cd ultralytics

/kaggle/working/ultralytics


In [ ]:
# Install the cloned Ultralytics package in editable mode.

!pip install -e . -q

<!-- # Import reusable libraries and define shared paths used throughout the notebook. -->

In [ ]:
import inspect
import os
import random
import shutil
import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

ULTRALYTICS_ROOT = "/kaggle/working/ultralytics"
ULTRALYTICS_PACKAGE = os.path.join(ULTRALYTICS_ROOT, "ultralytics")
BLOCK_PATH = os.path.join(ULTRALYTICS_PACKAGE, "nn", "modules", "block.py")
INIT_PATH = os.path.join(ULTRALYTICS_PACKAGE, "nn", "modules", "__init__.py")
TASKS_PATH = os.path.join(ULTRALYTICS_PACKAGE, "nn", "tasks.py")
YOLO11_YAML = os.path.join(ULTRALYTICS_PACKAGE, "cfg", "models", "11", "yolo11.yaml")
YOLO11_MCVAM_YAML = os.path.join(ULTRALYTICS_PACKAGE, "cfg", "models", "11", "yolo11_mcvam.yaml")
YOLO11S_MCVAM_YAML = os.path.join(ULTRALYTICS_PACKAGE, "cfg", "models", "11", "yolo11s_mcvam.yaml")

DATASET_PATH = "/kaggle/input/datasets/alimaged10/pcb-dataset/PCB-DATASET-master"
LABELS_DIR = "/kaggle/working/pcb_yolo_labels"
YOLO_ROOT = "/kaggle/working/PCB_YOLO"
MCVAM_PROJECT = "/kaggle/working/PCB_MCVAM"
MCVAM_RUN_NAME = "YOLO11s_MCVAM"

CLASSES = [
    "Missing_hole",
    "Mouse_bite",
    "Open_circuit",
    "Short",
    "Spur",
    "Spurious_copper"
]

CLASS_TO_ID = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

CLASS_NAMES = {
    idx: cls
    for idx, cls in enumerate(CLASSES)
}

# Lowercase aliases preserve the original notebook style without repeating declarations.
dataset_path = DATASET_PATH
labels_dir = LABELS_DIR
yolo_root = YOLO_ROOT
classes = CLASSES
class_to_id = CLASS_TO_ID
class_names = CLASS_NAMES
block_path = BLOCK_PATH
init_path = INIT_PATH
tasks_path = TASKS_PATH

print(ultralytics.__file__)

In [ ]:
# Run this notebook step.

import ultralytics

print(ultralytics.__file__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/kaggle/working/ultralytics/ultralytics/__init__.py


<!-- ## Phase 1: Add the MCVAM Attention Module to Ultralytics -->


In [ ]:
# Define the MCVAM module source code and append it to Ultralytics block.py.

mcvam_code = '''

class MCVAM(nn.Module):
    """
    Multi-Cognitive Visual Attention Module
    Inspired by PCB-MMF paper.
    """

    def __init__(self, c1, c2=None, *args, **kwargs):

        super().__init__()

        c2 = c1 if c2 is None else c2

        assert c1 == c2

        self.norm = nn.BatchNorm2d(c1)

        self.reduce = nn.Conv2d(
            c1,
            c1,
            kernel_size=1,
            bias=False
        )

        self.dw3 = nn.Conv2d(
            c1,
            c1,
            kernel_size=3,
            padding=1,
            groups=c1,
            bias=False
        )

        self.dw5 = nn.Conv2d(
            c1,
            c1,
            kernel_size=5,
            padding=2,
            groups=c1,
            bias=False
        )

        self.dw7 = nn.Conv2d(
            c1,
            c1,
            kernel_size=7,
            padding=3,
            groups=c1,
            bias=False
        )

        self.fuse = nn.Conv2d(
            c1,
            c1,
            kernel_size=1,
            bias=False
        )

        self.act = nn.SiLU()

    def forward(self, x):

        residual = x

        x = self.norm(x)

        x = self.reduce(x)

        b1 = self.dw3(x)
        b2 = self.dw5(x)
        b3 = self.dw7(x)

        x = (b1 + b2 + b3) / 3.0

        x = self.fuse(x)

        x = self.act(x)

        return x + residual

'''

with open(block_path, "r") as f:
    txt = f.read()

txt = txt.replace(
    "class SCDown(nn.Module):",
    mcvam_code + "\n\nclass SCDown(nn.Module):"
)

with open(block_path, "w") as f:
    f.write(txt)

print("MCVAM added successfully")

MCVAM added successfully


In [ ]:
# Verify that the MCVAM class was added to block.py exactly once.

with open(
    BLOCK_PATH,
    "r"
) as f:

    txt = f.read()

print("MCVAM" in txt)
print(txt.count("class MCVAM"))

True
1


In [ ]:
# Run this notebook step.

with open(init_path, "r") as f:
    txt = f.read()

txt = txt.replace(
    "C2PSA,",
    "C2PSA,\n    MCVAM,"
)

txt = txt.replace(
    '"C2PSA",',
    '"C2PSA",\n    "MCVAM",'
)

with open(init_path, "w") as f:
    f.write(txt)

print("MCVAM registered in __init__.py")

MCVAM registered in __init__.py


In [ ]:
# Count MCVAM registrations in the modules package initializer.

with open(
    INIT_PATH,
    "r"
) as f:
    txt = f.read()

print(txt.count("MCVAM"))

2


In [ ]:
# Run this notebook step.

with open(tasks_path, "r") as f:
    txt = f.read()

txt = txt.replace(
    "C2PSA,",
    "C2PSA,\n    MCVAM,"
)

with open(tasks_path, "w") as f:
    f.write(txt)

print("MCVAM imported into tasks.py")

MCVAM imported into tasks.py


In [ ]:
# Run this notebook step.

with open(
    TASKS_PATH,
    "r"
) as f:
    txt = f.read()

print(txt.count("MCVAM"))

3


In [ ]:
# Run this notebook step.

with open(tasks_path, "r") as f:
    txt = f.read()

txt = txt.replace(
    "C2PSA,\n            DWConv,",
    "C2PSA,\n            MCVAM,\n            DWConv,"
)

with open(tasks_path, "w") as f:
    f.write(txt)

print("MCVAM added to base_modules")

MCVAM added to base_modules


In [ ]:
# Run this notebook step.

with open(
    TASKS_PATH,
    "r"
) as f:
    txt = f.read()

print(txt.count("MCVAM"))

3


In [ ]:
# Print task parser lines containing MCVAM for verification.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "MCVAM" in line:
        print(i, line.strip())

18 MCVAM,
1687 MCVAM,
1730 MCVAM,


In [ ]:
# Add MCVAM to Ultralytics task parsing imports.

with open(TASKS_PATH, "r") as f:
    txt = f.read()

txt = txt.replace(
    "            MCVAM,\n            A2C2f,",
    "            A2C2f,"
)

with open(TASKS_PATH, "w") as f:
    f.write(txt)

print("Removed MCVAM from repeat_modules")

Removed MCVAM from repeat_modules


In [ ]:
# Print task parser lines containing MCVAM for verification.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "MCVAM" in line:
        print(i, line.strip())

18 MCVAM,
1687 MCVAM,
1730 MCVAM,


In [ ]:
# Retrieve the newly registered MCVAM class from Ultralytics and print it.

MCVAM = getattr(__import__("ultralytics.nn.modules", fromlist=["MCVAM"]), "MCVAM")

print(MCVAM)

<class 'ultralytics.nn.modules.block.MCVAM'>


In [ ]:
# Inspect the relevant tasks.py parser region around the module lists.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i in range(1675, 1735):
    print(f"{i}: {lines[i]}", end="")

In [ ]:
# Add MCVAM to Ultralytics task parsing imports.

with open(TASKS_PATH, "r") as f:
    txt = f.read()

txt = txt.replace(
"""            C2PSA,
    MCVAM,
            A2C2f,""",
"""            C2PSA,
            A2C2f,"""
)

with open(TASKS_PATH, "w") as f:
    f.write(txt)

print("Fixed repeat_modules")

Fixed repeat_modules


In [ ]:
# Print task parser lines containing MCVAM for verification.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "MCVAM" in line:
        print(i, line.strip())

18 MCVAM,
1687 MCVAM,


<!-- ## Phase 2: Create and Validate the YOLO11-MCVAM Model YAML -->


In [ ]:
# Search the cloned repository for YOLO11 model YAML files.

for root, dirs, files in os.walk("/kaggle/working/ultralytics"):
    for file in files:
        if "yolo11" in file.lower():
            print(os.path.join(root, file))

/kaggle/working/ultralytics/docs/en/models/yolo11.md
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11-pose.yaml
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11-cls-resnet18.yaml
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11.yaml
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11-seg.yaml
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11-obb.yaml
/kaggle/working/ultralytics/ultralytics/cfg/models/11/yolo11-cls.yaml
/kaggle/working/ultralytics/examples/YOLO-Axelera-Python/yolo11-seg.py


In [ ]:
# Print the original YOLO11 YAML configuration.

with open(YOLO11_YAML, "r") as f:
    print(f.read())

In [ ]:
# Copy the original YOLO11 YAML to a new MCVAM variant file.

shutil.copy(YOLO11_YAML, YOLO11_MCVAM_YAML)

print("Copied successfully")

Copied successfully


In [ ]:
# Insert the MCVAM block into the copied YOLO11 model YAML.

with open(YOLO11_MCVAM_YAML, "r") as f:
    txt = f.read()

txt = txt.replace(
    "- [-1, 2, C2PSA, [1024]] # 10",
    "- [-1, 2, C2PSA, [1024]] # 10\n  - [-1, 1, MCVAM, [1024]] # 11"
)

with open(YOLO11_MCVAM_YAML, "w") as f:
    f.write(txt)

print("MCVAM inserted")

MCVAM inserted


In [ ]:
# Print the YAML section around SPPF to confirm MCVAM placement.

with open(
    YOLO11_MCVAM_YAML,
    "r"
) as f:
    txt = f.read()

start = txt.find("SPPF")

print(txt[start:start+300])

In [ ]:
# Instantiate the YOLO11-MCVAM model from the custom YAML.

model = YOLO(
    YOLO11_MCVAM_YAML
)

print("Model created successfully")

WARNING ⚠️ no model scale passed. Assuming scale='n'.


Model created successfully


In [ ]:
# Inspect the relevant tasks.py parser region around the module lists.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i in range(1715, 1735):
    print(f"{i}: {lines[i]}", end="")

In [ ]:
# Inspect the relevant tasks.py parser region around the module lists.

with open(TASKS_PATH, "r") as f:
    lines = f.readlines()

for i in range(1744, 1765):
    print(f"{i}: {lines[i]}", end="")

In [ ]:
# Inspect the MCVAM constructor signature.

print(inspect.signature(MCVAM.__init__))

(self, c1, c2=None, *args, **kwargs)


In [ ]:
# Print the registered MCVAM source code for verification.

print(inspect.getsource(MCVAM))

In [ ]:
# Print the beginning of the MCVAM class definition from block.py.

with open(block_path, "r") as f:
    txt = f.read()

start = txt.find("class MCVAM")
print(txt[start:start+200])

class MCVAM(nn.Module):
    """
    Multi-Cognitive Visual Attention Module
    Inspired by PCB-MMF paper.
    """

    def __init__(self, c1, c2=None, *args, **kwargs):

        super().__init__()

 


In [ ]:
# Inspect the MCVAM constructor signature.

print(inspect.signature(MCVAM.__init__))

(self, c1, c2=None, *args, **kwargs)


In [ ]:
# Instantiate the YOLO11-MCVAM model from the custom YAML.

model = YOLO(
    YOLO11_MCVAM_YAML
)

print("SUCCESS")

WARNING ⚠️ no model scale passed. Assuming scale='n'.


SUCCESS


In [ ]:
# Print the constructed model architecture.

print(model.model)

In [ ]:
# Count total and trainable parameters in the current model.

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

print(f"Total Params: {total_params:,}")
print(f"Trainable Params: {trainable_params:,}")

Total Params: 3,655,616
Trainable Params: 3,655,600


In [ ]:
# Load the baseline YOLO11s model and count its parameters.

baseline = YOLO("yolo11s.pt")

baseline_params = sum(
    p.numel()
    for p in baseline.model.parameters()
)

print(f"Baseline Params: {baseline_params:,}")

In [ ]:
# Compare the parameter count added by the MCVAM variant.

print(
    "Added Parameters:",
    total_params - baseline_params
)

Added Parameters: -5803136


In [ ]:
# Run a dummy forward pass to confirm the custom model executes.

x = torch.randn(
    1,
    3,
    800,
    800
)

with torch.no_grad():
    y = model.model(x)

print("Forward pass successful")

Forward pass successful


<!-- ## Phase 3: Prepare the PCB Dataset and Train the MCVAM Model -->


<!-- # model training -->

In [ ]:
# Print Torch, CUDA, and GPU availability before training.

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA: True


GPU: Tesla T4


In [ ]:
# Instantiate the YOLO11-MCVAM model from the custom YAML.

model = YOLO(
    YOLO11_MCVAM_YAML
)

print(model.model)

In [ ]:
# Convert PCB XML annotations into YOLO-format label files.

# Paths
# ==========================================================

output_labels = LABELS_DIR

os.makedirs(output_labels, exist_ok=True)

# ==========================================================
# Class Mapping
# ==========================================================

class_to_id = {
    cls: idx
    for idx, cls in enumerate(classes)
}

# ==========================================================
# Conversion
# ==========================================================

num_images = 0
num_boxes = 0

for cls in classes:

    ann_dir = os.path.join(
        dataset_path,
        "Annotations",
        cls
    )

    img_dir = os.path.join(
        dataset_path,
        "images",
        cls
    )

    class_id = class_to_id[cls]

    for xml_file in os.listdir(ann_dir):

        if not xml_file.endswith(".xml"):
            continue

        xml_path = os.path.join(
            ann_dir,
            xml_file
        )

        image_name = xml_file.replace(
            ".xml",
            ".jpg"
        )

        image_path = os.path.join(
            img_dir,
            image_name
        )

        if not os.path.exists(image_path):
            continue

        # Read image dimensions
        img = cv2.imread(image_path)

        if img is None:
            continue

        h, w = img.shape[:2]

        tree = ET.parse(xml_path)
        root = tree.getroot()

        yolo_lines = []

        for obj in root.findall("object"):

            bbox = obj.find("bndbox")

            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # YOLO format
            x_center = ((xmin + xmax) / 2) / w
            y_center = ((ymin + ymax) / 2) / h

            box_w = (xmax - xmin) / w
            box_h = (ymax - ymin) / h

            yolo_lines.append(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_w:.6f} "
                f"{box_h:.6f}"
            )

            num_boxes += 1

        label_path = os.path.join(
            output_labels,
            image_name.replace(".jpg", ".txt")
        )

        with open(label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        num_images += 1

print(f"Converted Images : {num_images}")
print(f"Converted Boxes  : {num_boxes}")
print(f"Labels Saved To  : {output_labels}")

Converted Images : 693
Converted Boxes  : 2953
Labels Saved To  : /kaggle/working/pcb_yolo_labels


In [ ]:
# Inspect generated YOLO label files and print a sample.

label_files = [
    f for f in os.listdir(labels_dir)
    if f.endswith(".txt")
]

print("Label Files:", len(label_files))

sample = sorted(label_files)[0]

print("\nSample Label File:")
print(sample)

with open(os.path.join(labels_dir, sample)) as f:
    print(f.read())

Label Files: 693

Sample Label File:
01_missing_hole_01.txt
0 0.822182 0.820618 0.023401 0.034678
0 0.542518 0.230139 0.021753 0.039092
0 0.580587 0.519546 0.023401 0.037831


In [ ]:
# Count all boxes in the generated YOLO label files.

total_boxes = 0

for file in os.listdir(labels_dir):

    if file.endswith(".txt"):

        with open(os.path.join(labels_dir, file)) as f:
            total_boxes += len(f.readlines())

print("Total Boxes:", total_boxes)

Total Boxes: 2953


In [ ]:
# Check whether image names are unique across class folders.

all_names = []

for cls in classes:

    img_dir = os.path.join(dataset_path, "images", cls)

    for img_name in os.listdir(img_dir):

        if img_name.endswith(".jpg"):
            all_names.append(img_name)

print("Total images:", len(all_names))
print("Unique names:", len(set(all_names)))

Total images: 693
Unique names: 693


In [ ]:
# Build a DataFrame of PCB image names, classes, and source paths.

records = []

for cls in classes:

    img_dir = os.path.join(
        dataset_path,
        "images",
        cls
    )

    for img_name in os.listdir(img_dir):

        if not img_name.lower().endswith(".jpg"):
            continue

        records.append([
            img_name,
            cls,
            os.path.join(img_dir, img_name)
        ])

df = pd.DataFrame(
    records,
    columns=[
        "image_name",
        "class",
        "image_path"
    ]
)

print("Total Images:", len(df))
df.head()

Total Images: 693


,image_name,class,image_path
0,01_missing_hole_01.jpg,Missing_hole,/kaggle/input/datasets/alimaged10/pcb-dataset/...
1,04_missing_hole_01.jpg,Missing_hole,/kaggle/input/datasets/alimaged10/pcb-dataset/...
2,01_missing_hole_17.jpg,Missing_hole,/kaggle/input/datasets/alimaged10/pcb-dataset/...
3,04_missing_hole_12.jpg,Missing_hole,/kaggle/input/datasets/alimaged10/pcb-dataset/...
4,07_missing_hole_06.jpg,Missing_hole,/kaggle/input/datasets/alimaged10/pcb-dataset/...


In [ ]:
# Split the image DataFrame into train, validation, and test subsets.

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("Total:", len(train_df)+len(val_df)+len(test_df))

Train: 485
Val  : 138
Test : 70
Total: 693


In [ ]:
# Create the YOLO image and label folder structure.

folders = [

    "images/train",
    "images/val",
    "images/test",

    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:

    os.makedirs(
        os.path.join(yolo_root, folder),
        exist_ok=True
    )

print("Folders created.")

Folders created.


In [ ]:
# Copy images and labels into the YOLO train, validation, and test folders.

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

for split_name, split_df in splits.items():

    for _, row in split_df.iterrows():

        image_name = row["image_name"]

        image_src = row["image_path"]

        label_src = os.path.join(
            labels_dir,
            image_name.replace(".jpg", ".txt")
        )

        image_dst = os.path.join(
            yolo_root,
            "images",
            split_name,
            image_name
        )

        label_dst = os.path.join(
            yolo_root,
            "labels",
            split_name,
            image_name.replace(".jpg", ".txt")
        )

        shutil.copy2(
            image_src,
            image_dst
        )

        shutil.copy2(
            label_src,
            label_dst
        )

print("Dataset copied.")

Dataset copied.


In [ ]:
# Verify image and label counts in each YOLO split.

for split in ["train","val","test"]:

    n_images = len(
        os.listdir(
            os.path.join(
                yolo_root,
                "images",
                split
            )
        )
    )

    n_labels = len(
        os.listdir(
            os.path.join(
                yolo_root,
                "labels",
                split
            )
        )
    )

    print(
        f"{split:<5} "
        f"Images={n_images} "
        f"Labels={n_labels}"
    )

train Images=485 Labels=485
val   Images=138 Labels=138
test  Images=70 Labels=70


In [ ]:
# Write the YOLO data.yaml file for training.

yaml_content = """
path: /kaggle/working/PCB_YOLO

train: images/train
val: images/val
test: images/test

names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper
"""

with open(
    os.path.join(YOLO_ROOT, "data.yaml"),
    "w"
) as f:

    f.write(yaml_content)

print("data.yaml created")

data.yaml created


In [ ]:
# Visualize random training images with YOLO labels drawn as boxes.

image_dir = os.path.join(
    yolo_root,
    "images",
    "train"
)

label_dir = os.path.join(
    yolo_root,
    "labels",
    "train"
)

samples = random.sample(
    os.listdir(image_dir),
    6
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18,10)
)

for ax, image_name in zip(
    axes.ravel(),
    samples
):

    image_path = os.path.join(
        image_dir,
        image_name
    )

    label_path = os.path.join(
        label_dir,
        image_name.replace(".jpg", ".txt")
    )

    img = cv2.imread(image_path)
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    with open(label_path) as f:

        lines = f.readlines()

    for line in lines:

        cls, xc, yc, bw, bh = map(
            float,
            line.strip().split()
        )

        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)

        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (255,0,0),
            3
        )

        cv2.putText(
            img,
            class_names[int(cls)],
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize one labeled example for each defect class.

image_dir = os.path.join(yolo_root, "images", "train")
label_dir = os.path.join(yolo_root, "labels", "train")

selected = {}

for img_name in os.listdir(image_dir):

    label_path = os.path.join(
        label_dir,
        img_name.replace(".jpg", ".txt")
    )

    with open(label_path) as f:

        lines = f.readlines()

    for line in lines:

        cls = int(line.split()[0])

        if cls not in selected:

            selected[cls] = img_name

        if len(selected) == 6:
            break

    if len(selected) == 6:
        break

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18,10)
)

for ax, cls in zip(
    axes.ravel(),
    sorted(selected.keys())
):

    img_name = selected[cls]

    img = cv2.imread(
        os.path.join(image_dir, img_name)
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    with open(
        os.path.join(
            label_dir,
            img_name.replace(".jpg", ".txt")
        )
    ) as f:

        lines = f.readlines()

    for line in lines:

        c, xc, yc, bw, bh = map(
            float,
            line.split()
        )

        x1 = int((xc - bw/2)*w)
        y1 = int((yc - bh/2)*h)
        x2 = int((xc + bw/2)*w)
        y2 = int((yc + bh/2)*h)

        cv2.rectangle(
            img,
            (x1,y1),
            (x2,y2),
            (255,0,0),
            3
        )

    ax.imshow(img)
    ax.set_title(class_names[cls])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Run this notebook step.

dataset_root = yolo_root

for split in ["train", "val", "test"]:

    image_dir = os.path.join(
        dataset_root,
        "images",
        split
    )

    label_dir = os.path.join(
        dataset_root,
        "labels",
        split
    )

    print(
        f"{split.upper():<6}",
        "Images:",
        len(os.listdir(image_dir)),
        "| Labels:",
        len(os.listdir(label_dir))
    )

TRAIN  Images: 485 | Labels: 485
VAL    Images: 138 | Labels: 138
TEST   Images: 70 | Labels: 70


<!-- ## Phase 4: Configure YOLO11s-MCVAM and Run Experiments -->


In [ ]:
# Insert the MCVAM block into the copied YOLO11 model YAML.

with open(YOLO11_MCVAM_YAML, "r") as f:
    txt = f.read()

txt = txt.replace(
    "nc: 80 # number of classes",
    "nc: 80 # number of classes\nscale: s"
)

with open(YOLO11_MCVAM_YAML, "w") as f:
    f.write(txt)

print("Scale set to s")

In [ ]:
# Run this notebook step.

with open(YOLO11_MCVAM_YAML, "r") as f:
    lines = f.readlines()

for i in range(15):
    print(lines[i], end="")

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs


In [ ]:
# Run this notebook step.

shutil.copy(YOLO11_MCVAM_YAML, YOLO11S_MCVAM_YAML)

print("Created yolo11s_mcvam.yaml")

Created yolo11s_mcvam.yaml


In [ ]:
# Instantiate the scaled YOLO11s-MCVAM model and print its scale.

model = YOLO(
    YOLO11S_MCVAM_YAML
)

print("Scale =", model.model.yaml.get("scale"))

Scale = s


In [ ]:
# Load YOLO11s pretrained weights into the custom MCVAM model.

model.load("yolo11s.pt")

In [ ]:
# Count total and trainable parameters in the current model.

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

print(f"Total Params: {total_params:,}")
print(f"Trainable Params: {trainable_params:,}")

Total Params: 14,353,344
Trainable Params: 14,353,328


In [ ]:

# Load YOLO11s pretrained weights into the custom MCVAM model.

# results = model.train(

#     data=os.path.join(YOLO_ROOT, "data.yaml"),

#     epochs=100,

#     imgsz=800,

#     batch=8,

#     workers=2,

#     seed=42,

#     patience=20,

#     optimizer="auto",

#     lr0=0.01,

#     fliplr=0.5,
#     flipud=0.5,

#     degrees=20,

#     translate=0.1,

#     scale=0.2,

#     mosaic=1.0,

#     mixup=0,

#     project=MCVAM_PROJECT,

#     name=MCVAM_RUN_NAME,

#     exist_ok=True
# )

# results = model.train(

#     data=os.path.join(YOLO_ROOT, "data.yaml"),

#     epochs=200,

#     imgsz=800,

#     batch=8,

#     workers=2,

#     seed=42,

#     patience=30,

#     optimizer="auto",

#     lr0=0.01,

#     fliplr=0.5,
#     flipud=0.5,

#     degrees=20,

#     translate=0.1,

#     scale=0.2,

#     mosaic=1.0,

#     mixup=0,

#     project=MCVAM_PROJECT,

#     name="YOLO11s_MCVAM_PRETRAINED",

#     exist_ok=True
# )

from itertools import product

# ==========================================
# Grid Search Parameters
# ==========================================

img_sizes = [640, 800]

learning_rates = [
    0.01
    0.001,
    0.005
]

batches = [
    8,
    16
]

# ==========================================
# Results Storage
# ==========================================

results_table = []

# ==========================================
# Grid Search
# ==========================================

for imgsz, lr, batch in product(
    img_sizes,
    learning_rates,
    batches
):

    print("\n" + "=" * 70)

    print(
        f"Running : "
        f"imgsz={imgsz} | "
        f"lr={lr} | "
        f"batch={batch}"
    )

    print("=" * 70)

    run_name = (
        f"img{imgsz}"
        f"_lr{lr}"
        f"_bs{batch}"
    )

    try:

        # --------------------------------------
        # Load Model
        # --------------------------------------

        model = YOLO(
            YOLO11S_MCVAM_YAML
        )

        model.load("yolo11s.pt")

        # --------------------------------------
        # Training
        # --------------------------------------

        model.train(

            data=os.path.join(YOLO_ROOT, "data.yaml"),

            epochs=300,
            patience=30,

            imgsz=imgsz,

            batch=batch,

            lr0=lr,

            workers=2,

            seed=42,

            fliplr=0.5,
            flipud=0.5,

            degrees=20,

            translate=0.1,

            scale=0.2,

            mosaic=1.0,

            mixup=0,

            project="/kaggle/working/GridSearch",

            name=run_name,

            exist_ok=True
        )

        # --------------------------------------
        # Validation
        # --------------------------------------

        best_model = YOLO(
            f"/kaggle/working/GridSearch/{run_name}/weights/best.pt"
        )

        metrics = best_model.val(
            data=os.path.join(YOLO_ROOT, "data.yaml")
        )

        # --------------------------------------
        # Read Training CSV
        # --------------------------------------

        csv_path = (
            f"/kaggle/working/GridSearch/"
            f"{run_name}/results.csv"
        )

        results_csv = pd.read_csv(csv_path)

        stopped_epoch = len(results_csv)

        best_epoch_map50 = int(
            results_csv["metrics/mAP50(B)"].idxmax()
        )

        best_epoch_map5095 = int(
            results_csv["metrics/mAP50-95(B)"].idxmax()
        )

        # --------------------------------------
        # Save Results
        # --------------------------------------

        results_table.append({

            "Run Name": run_name,

            "Image Size": imgsz,

            "Learning Rate": lr,

            "Batch Size": batch,

            "Stopped Epoch": stopped_epoch,

            "Best Epoch mAP50":
                best_epoch_map50,

            "Best Epoch mAP50-95":
                best_epoch_map5095,

            "mAP50":
                metrics.box.map50,

            "mAP50-95":
                metrics.box.map,

            "Precision":
                metrics.box.mp,

            "Recall":
                metrics.box.mr

        })

        # --------------------------------------
        # Update Excel After Every Run
        # --------------------------------------

        df = pd.DataFrame(results_table)

        df = df.sort_values(
            by="mAP50-95",
            ascending=False
        )

        df.to_excel(
            "/kaggle/working/Grid_Search_Results.xlsx",
            index=False
        )

        print("\nCurrent Top Results:\n")

        print(df.head())

    except Exception as e:

        print(
            f"\nFAILED RUN: {run_name}"
        )

        print(e)

# ==========================================
# Final Save
# ==========================================

df = pd.DataFrame(results_table)

df = df.sort_values(
    by="mAP50-95",
    ascending=False
)

df.to_excel(
    "/kaggle/working/Grid_Search_Results.xlsx",
    index=False
)

print("\nGrid Search Complete!")

print(
    "\nExcel Saved To:\n"
    "/kaggle/working/Grid_Search_Results.xlsx"
)

print("\nBest Configuration:\n")

print(df.iloc[0])

In [ ]:
# Count all boxes in the generated YOLO label files.

label_dir = os.path.join(YOLO_ROOT, "labels", "train")

total_boxes = 0

for f in os.listdir(label_dir):
    if f.endswith(".txt"):
        with open(os.path.join(label_dir, f)) as file:
            total_boxes += len(file.readlines())

print("Total boxes:", total_boxes)

In [ ]:
# Validate the trained model and print detection metrics.

# results = model.val()

# print(results.box.map)
# print(results.box.map50)
# print(results.box.map75)

In [ ]:
# Keep this experimental or reference cell documented without running it.

# Image(
#     os.path.join(MCVAM_PROJECT, MCVAM_RUN_NAME, "results.png"),
#     width=1200
# )

In [ ]:
# Keep this experimental or reference cell documented without running it.

# Image(
#     os.path.join(MCVAM_PROJECT, MCVAM_RUN_NAME, "confusion_matrix.png"),
#     width=900
# )

In [ ]:
# Validate the trained model and print detection metrics.

#     os.path.join(MCVAM_PROJECT, MCVAM_RUN_NAME, "weights", "best.pt")
# )

# metrics = best_model.val(
#     data=os.path.join(YOLO_ROOT, "data.yaml"),
#     split="test"
# )

# print("mAP50     :", metrics.box.map50)
# print("mAP50-95  :", metrics.box.map)
# print("Precision :", metrics.box.mp)
# print("Recall    :", metrics.box.mr)

In [ ]:
# Keep this experimental or reference cell documented without running it.

# import random
# import matplotlib.pyplot as plt
# import cv2

# best_model = YOLO(
#     os.path.join(MCVAM_PROJECT, MCVAM_RUN_NAME, "weights", "best.pt")
# )

# test_dir = os.path.join(YOLO_ROOT, "images", "test")

# images = random.sample(
#     os.listdir(test_dir),
#     20
# )

# fig, axes = plt.subplots(
#     5,
#     4,
#     figsize=(16,20)
# )

# for ax, img_name in zip(axes.flatten(), images):

#     img_path = os.path.join(
#         test_dir,
#         img_name
#     )

#     results = best_model.predict(
#         img_path,
#         conf=0.25,
#         verbose=False
#     )

#     plotted = results[0].plot()

#     plotted = cv2.cvtColor(
#         plotted,
#         cv2.COLOR_BGR2RGB
#     )

#     ax.imshow(plotted)
#     ax.set_title(img_name, fontsize=8)
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# Archive the trained MCVAM experiment folder for download.

# shutil.make_archive(
#     "/kaggle/working/YOLO11_MCVAM",
#     "zip",
#     MCVAM_PROJECT
# )

# print("Saved")